In [1]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
PROC_DIR = "../datasets_processed"
test_df = pd.read_csv("../datasets/test.csv")

In [3]:
class SignDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.load(f"{PROC_DIR}/{row.file}").astype(np.float32)
        y = int(row.label)
        return torch.tensor(x), torch.tensor(y)

    def __len__(self):
        return len(self.df)

In [4]:
test_loader = DataLoader(SignDataset(test_df), batch_size=32)

In [6]:
# Load model
input_dim = (54+21+21+33)*3
num_classes = test_df.label.nunique()

class LSTMModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_dim, 256, batch_first=True, num_layers=2)
        self.fc = torch.nn.Linear(256, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

model = LSTMModel()
model.load_state_dict(torch.load("../Models/lstm_bdslw401.pth", map_location="cpu"))
model.eval()

LSTMModel(
  (lstm): LSTM(387, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=401, bias=True)
)

In [7]:
# Predict
all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_loader:
        preds = model(X)
        all_preds.extend(preds.argmax(1).numpy())
        all_labels.extend(y.numpy())

print(classification_report(all_labels, all_preds))


ValueError: invalid literal for int() with base 10: 'W269'

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(16,16))
sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
plt.title("Confusion Matrix")
plt.show()
